# 将shp中的中文翻译成英文

## 读取数据

In [ ]:
import geopandas as gpd
import re

# pip install pypinyin
from pypinyin import lazy_pinyin

# 1) 读城市边界 GeoJSON
city_gdf = gpd.read_file("/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市.geojson")

In [ ]:
city_gdf

## 提取线数据，因为无法保存

In [ ]:
# 只保留 Polygon / MultiPolygon
poly_gdf = city_gdf[city_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# 看看被剔除的是哪些（可选）
bad = city_gdf[~city_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
print("Non-polygon geometries:")
print(bad[["name"]].assign(geom_type=bad.geometry.geom_type))

In [ ]:
# 2) 中文行政后缀清理（按需要可继续加）
SUFFIX_PATTERN = r"(市|地区|盟|自治州|自治县|县|区|旗|自治旗|林区|特区|保护区|省|自然保护区|特别行政区|地区)$"

def cn2eng_pinyin(name_cn: str) -> str:
    if name_cn is None:
        return None
    s = str(name_cn).strip()
    # 去掉末尾常见行政后缀
    s = re.sub(SUFFIX_PATTERN, "", s)
    # 转拼音（不带声调），并把每个音节首字母大写后拼接
    py = lazy_pinyin(s)
    eng = "".join([p.capitalize() for p in py])
    return eng

poly_gdf["name_eng"] = poly_gdf["name"].apply(cn2eng_pinyin)

In [ ]:
poly_gdf

In [ ]:
# 3) 导出 Shapefile
out_path = "/Users/wl/Paper_First/202601交通数据集LLM/data/figure/china_geodata/中国_市_eng.shp"

# Shapefile对字段名长度有限制（通常<=10），name_eng刚好8没问题
# 注意：Shapefile对编码敏感，写入时指定UTF-8更稳（不同GDAL版本表现略有差异）
poly_gdf.to_file(out_path, driver="ESRI Shapefile", encoding="utf-8")

print("Done! Saved to:", out_path)
print(poly_gdf[["name", "name_eng"]])